# Deteccao de Outliers em Series Temporais Financeiras com Autoencoder Simples
## e Agente de Decisao de Compra e Venda

**Disciplina:** Projeto de Deep Learning

**Curso:** Ciencia de Dados e IA

**Professor(a):** Renan Santos Mendes

**Email:** renansantosmendes@gmail.com

---

Este notebook e uma variacao do notebook `autoencoder_lstm_outlier_trading_agent_v3_draft`,
com as seguintes diferencas principais:

1. O autoencoder recorrente (`LSTMAutoencoder`) e substituido por um
   autoencoder **totalmente conectado (denso)**, mais simples, sem
   nenhuma camada `LSTM`.
2. Em vez de retornos simulados por um processo GARCH, o autoencoder e
   treinado sobre os **log-retornos historicos reais** de um ativo,
   coletados com `fetch_price_series` (via `yfinance`) e calculados
   diretamente em tensores (`torch.log`), sem depender de uma funcao
   auxiliar baseada em `numpy`.
3. Uma parte dos dados e separada para **validacao**: o autoencoder e
   treinado apenas sobre o conjunto de treino, e o erro de
   reconstrucao no conjunto de validacao e reportado como verificacao
   de generalizacao.
4. As janelas deslizantes sao construidas com a classe
   `SlidingWindowReconstructionDataset`, de
   `generative_models.data.datasets`, em vez de uma funcao manual, e
   consumidas por meio de um `DataLoader` com mini-batches. Os
   tensores normalizados sao passados diretamente a essa classe, sem
   a conversao intermediaria para `numpy`.
5. O experimento e rastreado com `WandbExperimentTracker`, seguindo o
   mesmo padrao de treinamento com barra de progresso (`tqdm`) e
   `log_metrics`/`log_model`/`finish_run` usado em
   `regression_log_return_training.ipynb`.

O pipeline permanece o mesmo do notebook original:

1. Download da serie de precos reais de um ativo via `yfinance` e
   calculo dos log-retornos diarios.
2. Construcao de um autoencoder denso simples (encoder/decoder com
   camadas `Linear` e `ReLU`).
3. Treinamento do autoencoder, com o experimento rastreado pelo
   `WandbExperimentTracker`, e calculo do erro de reconstrucao como
   escore de anomalia.
4. Utilizacao de um gerador de sinais (`TradingSignalGenerator`),
   importado de `pgl_utils.deep_learning`, que converte os outliers
   detectados em sinais de compra e venda.
5. Visualizacao dos outliers detectados e das oportunidades de compra
   e venda identificadas pelo gerador de sinais, sobre a serie de
   precos real.

Como os dados agora sao reais (e nao mais simulados com choques
conhecidos), a secao de avaliacao frente a um gabarito sintetico do
notebook original nao se aplica aqui e foi removida.

## Instalacao das dependencias

A celula abaixo instala, por meio do `uv`, as bibliotecas necessarias
para a execucao do notebook em um ambiente Google Colab. Caso as
bibliotecas ja estejam disponiveis no ambiente, a instalacao e
ignorada automaticamente pelo `uv`.

In [ ]:
!uv pip install yfinance wandb python-dotenv pgl-utils

## Importacao das bibliotecas

A celula a seguir importa todas as bibliotecas utilizadas ao longo do
notebook, incluindo as classes e funcoes proprias do projeto
(`generative_models` e `pgl_utils`) reaproveitadas de
`regression_log_return_training.ipynb`, e fixa as sementes de
aleatoriedade, garantindo a reprodutibilidade dos resultados.

In [ ]:
import pgl_utils

In [ ]:
pgl_utils.__version__

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from dotenv import load_dotenv
from tqdm.auto import tqdm

from pgl_utils.deep_learning import (
    plot_loss_curve,
    plot_time_series,
    plot_reconstruction_error_with_threshold,
    plot_outlier_detection_and_trading_signals,
    TradingSignalGenerator,
)

from generative_models.common.normalization import normalize_series
from generative_models.data.datasets import SlidingWindowReconstructionDataset
from generative_models.data.market_data import fetch_price_series
from generative_models.tracking.wandb_tracker import WandbExperimentTracker

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## Chave de API do wandb

Em ambiente Colab, a chave `WANDB_API_KEY` e lida dos `Secrets` do
Colab e definida na variavel de ambiente antes de o
`WandbExperimentTracker` ser instanciado. Fora do Colab, o proprio
`WandbExperimentTracker` cuida do carregamento da chave a partir de um
arquivo `.env` local, por meio de `python-dotenv`.

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import userdata

    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
else:
    load_dotenv()

## Parametros do experimento e coleta de precos reais

As constantes abaixo controlam o experimento (ticker, intervalo de
datas) e sao usadas em todo o notebook, inclusive na configuracao
enviada ao wandb. Os precos de fechamento diarios reais sao coletados
com `fetch_price_series`, que encapsula o `yfinance` e retorna um
dataframe de uma unica coluna (`close_price`).

In [ ]:
from pgl_utils.deep_learning import load_brazil_tickers, load_us_tickers

brazil_tickers = load_brazil_tickers()
us_tickers = load_us_tickers()

print(list(brazil_tickers.keys()))
# ['petroleo_gas_energia', 'mineracao_siderurgia_materiais', 'bancos_servicos_financeiros', ...]

print(brazil_tickers["industria_tecnologia"])
# ['PETR3.SA', 'PETR4.SA', 'PRIO3.SA', 'RECV3.SA', 'CSAN3.SA']


In [ ]:
TICKER = "AAPL"
START_DATE = "2020-01-01"
END_DATE = "2026-09-15"

price_data = fetch_price_series(TICKER, START_DATE, END_DATE)

A visualizacao abaixo mostra a serie de precos de fechamento reais do
ativo escolhido, utilizada como base para o restante do notebook.

In [ ]:
price_figure = plot_time_series(
    price_data,
    "close_price",
    f"{TICKER} - serie de precos de fechamento (dados reais)",
)
price_figure.show()

## Construcao das series de log-retorno e preco

O preco de fechamento e convertido diretamente em um tensor
(`price_tensor`), e o log-retorno e calculado com operacoes de tensor
(`torch.log`): `log_return_tensor[t] = log(price_tensor[t + 1]) -
log(price_tensor[t])`. Em vez de reunir data, retorno e preco em um
unico `DataFrame`, cada serie e mantida em sua propria variavel
(`dates`, `return_values`, `price_values`), todas alinhadas pelo
indice e reutilizadas separadamente mais adiante, tanto para a
deteccao de outliers quanto para a visualizacao dos sinais de compra e
venda.

In [ ]:
price_tensor = torch.tensor(
    price_data["close_price"].to_numpy(), dtype=torch.float32
)
log_return_tensor = torch.log(price_tensor[1:]) - torch.log(price_tensor[:-1])

dates = price_data.index[1:]
price_values = price_tensor[1:].numpy()
return_values = log_return_tensor.numpy()
number_of_observations = len(dates)

print(
    f"Ticker: {TICKER} | Observacoes: {number_of_observations} | "
    f"Periodo: {dates.min().date()} a {dates.max().date()}"
)

## Separacao treino/validacao e normalizacao

Os log-retornos sao divididos cronologicamente em treino (80%) e
validacao (20%), sem embaralhamento, para nao vazar informacao do
futuro para o passado. A normalizacao (media e desvio padrao) e
calculada apenas a partir do conjunto de treino, com
`normalize_series`, e a mesma estatistica e aplicada ao conjunto de
validacao e a serie completa, evitando vazamento de dados.

In [ ]:
train_split_index = int(len(log_return_tensor) * 0.8)

train_log_return_tensor = log_return_tensor[:train_split_index]
validation_log_return_tensor = log_return_tensor[train_split_index:]

normalized_train_return_tensor, return_train_mean, return_train_std = (
    normalize_series(train_log_return_tensor)
)
normalized_validation_return_tensor, _, _ = normalize_series(
    validation_log_return_tensor,
    mean=return_train_mean,
    std=return_train_std,
)
normalized_full_return_tensor, _, _ = normalize_series(
    log_return_tensor,
    mean=return_train_mean,
    std=return_train_std,
)

## Construcao das janelas deslizantes

`SlidingWindowReconstructionDataset`, de
`generative_models.data.datasets`, gera uma janela por amostra: cada
janela e, ao mesmo tempo, a entrada do autoencoder e o alvo que o
decodificador precisa reconstruir. Sao criados tres datasets a partir
da mesma classe: um para treino, um para validacao (ambos usados no
laco de treinamento) e um sobre a serie completa e cronologica
(treino seguido de validacao), usado mais adiante para calcular o
escore de anomalia de cada janela ao longo de toda a linha do tempo.

In [ ]:
WINDOW_SIZE = 10
BATCH_SIZE = 32
CONTAMINATION = 0.05

train_window_dataset = SlidingWindowReconstructionDataset(
    series_values=normalized_train_return_tensor,
    window_size=WINDOW_SIZE,
)
validation_window_dataset = SlidingWindowReconstructionDataset(
    series_values=normalized_validation_return_tensor,
    window_size=WINDOW_SIZE,
)
full_window_dataset = SlidingWindowReconstructionDataset(
    series_values=normalized_full_return_tensor,
    window_size=WINDOW_SIZE,
)

dataloader_generator = torch.Generator()
dataloader_generator.manual_seed(RANDOM_SEED)

train_window_loader = DataLoader(
    train_window_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=dataloader_generator,
)
validation_window_loader = DataLoader(
    validation_window_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

window_end_indices = np.arange(WINDOW_SIZE - 1, number_of_observations)

print(
    f"Janelas de treino: {len(train_window_dataset)} | "
    f"Janelas de validacao: {len(validation_window_dataset)}"
)

## Autoencoder denso simples

Diferentemente de um autoencoder recorrente (`LSTM`), que processa a
janela como uma sequencia temporal, o `SimpleAutoencoder` trata cada
janela de retornos como um vetor de features independentes, sendo
composto apenas por camadas `Linear` com ativacao `ReLU`. O codificador
comprime a janela de tamanho `WINDOW_SIZE` em um vetor latente de
dimensao `latent_dim`; o decodificador reconstroi a janela original a
partir desse vetor latente.

In [ ]:
class SimpleAutoencoder(nn.Module):
    """A simple fully connected (dense) autoencoder.

    The network is composed of an encoder that compresses the input
    window into a lower-dimensional latent representation, and a
    decoder that reconstructs the original window from that
    representation. Unlike a recurrent autoencoder, every position of
    the window is treated as an independent feature.

    Attributes:
        encoder: The sequential module that encodes the input into
            the latent space.
        decoder: The sequential module that reconstructs the input
            from the latent space.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 16,
        latent_dim: int = 4,
    ) -> None:
        """Initialize the encoder and decoder layers.

        Args:
            input_dim: The dimensionality of the input window
                (number of returns per window).
            hidden_dim: The number of units in the hidden layers.
            latent_dim: The dimensionality of the latent
                representation.
        """
        super().__init__()
        self.encoder = nn.Sequential(
            ...
        )
        self.decoder = nn.Sequential(
            ...
        )

    def forward(self, input_tensor: torch.Tensor) -> torch.Tensor:
        """Run the forward pass through the autoencoder.

        Args:
            input_tensor: A tensor of shape (batch_size, input_dim)
                containing the input windows.

        Returns:
            A tensor of shape (batch_size, input_dim) containing the
            reconstructed windows.
        """
        ...

A celula abaixo instancia o `SimpleAutoencoder` com o tamanho de
janela definido anteriormente e o move para a `device` disponivel
(GPU, quando existente).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
hidden_layer_size = 16
latent_layer_size = 4

dense_model = ...
print(dense_model)

A funcao `compute_reconstruction_scores` calcula, para cada janela, o
erro medio de reconstrucao do autoencoder ja treinado, utilizado como
escore de anomalia: quanto maior o erro, mais dificil foi para o
modelo reconstruir aquela janela, o que sugere um padrao incomum nos
retornos reais do ativo.

In [ ]:
def compute_reconstruction_scores(
    model: nn.Module,
    input_tensor: torch.Tensor,
) -> np.ndarray:
    """Compute per-window reconstruction error scores.

    Args:
        model: A trained autoencoder model.
        input_tensor: A tensor containing the windows to be scored,
            of shape (num_windows, window_size).

    Returns:
        A one-dimensional numpy array with the mean squared
        reconstruction error of each window.
    """
    criterion = nn.MSELoss(reduction="none")
    model.eval()
    with torch.no_grad():
        reconstruction = model(input_tensor)
        errors = criterion(reconstruction, input_tensor).mean(dim=1)
    return errors.cpu().numpy()

## Rastreamento do experimento com o wandb

`WandbExperimentTracker` centraliza a tentativa de login, a
inicializacao do run e o registro de metricas no Weights & Biases. O
dicionario `config` documenta o ativo, o periodo, os hiperparametros
das janelas e do treinamento, para reprodutibilidade do experimento.

In [ ]:
number_of_epochs = 300
learning_rate = 1e-3

experiment_tracker = WandbExperimentTracker(
    project_name="autoencoder-simple-outlier-trading-agent",
    config={
        "ticker": TICKER,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "window_size": WINDOW_SIZE,
        "batch_size": BATCH_SIZE,
        "contamination": CONTAMINATION,
        "epochs": number_of_epochs,
        "learning_rate": learning_rate,
        "hidden_layer_size": hidden_layer_size,
        "latent_layer_size": latent_layer_size,
        "optimizer": "Adam",
        "loss_function": "MSE",
        "model": "SimpleAutoencoder",
        "random_seed": RANDOM_SEED,
    },
)

O laco de treinamento abaixo otimiza o erro quadratico medio (MSE) de
reconstrucao das janelas de treino, usando o otimizador Adam e
mini-batches fornecidos por `train_window_loader`. A cada epoca, a
perda media e registrada em `training_loss_history` e no wandb, por
meio de `experiment_tracker.log_metrics`, e exibida na barra de
progresso do `tqdm`. Ao final do treinamento, o erro de reconstrucao
no conjunto de validacao (nunca visto durante o treino) e calculado e
tambem registrado, o modelo treinado e salvo como artifact com
`log_model`, o run e encerrado com `finish_run`, e a curva de
aprendizado e exibida com `plot_loss_curve`.

In [ ]:
experiment_tracker.start_run()

loss_function = nn.MSELoss()
optimizer = optim.Adam(dense_model.parameters(), lr=learning_rate)

training_loss_history = []
epoch_progress_bar = tqdm(range(number_of_epochs), desc="Training", unit="epoch")

for epoch_index in epoch_progress_bar:
    dense_model.train()
    epoch_loss_total = 0.0

    for batch_windows in train_window_loader:
        ...
        
        epoch_loss_total += loss_value.item() * batch_windows.shape[0]

    epoch_average_loss = epoch_loss_total / len(train_window_dataset)
    training_loss_history.append(epoch_average_loss)

    experiment_tracker.log_metrics(
        {"epoch": epoch_index + 1, "train/loss_mse": epoch_average_loss}
    )
    epoch_progress_bar.set_postfix(loss=f"{epoch_average_loss:.6f}")

dense_model.eval()
validation_loss_total = 0.0
with torch.no_grad():
    for batch_windows in validation_window_loader:
        batch_windows = batch_windows.to(device)
        reconstruction = dense_model(batch_windows)
        loss_value = loss_function(reconstruction, batch_windows)
        validation_loss_total += loss_value.item() * batch_windows.shape[0]

validation_average_loss = validation_loss_total / len(validation_window_dataset)
experiment_tracker.log_metrics({"validation/loss_mse": validation_average_loss})
print(f"Validation MSE: {validation_average_loss:.6f}")

experiment_tracker.log_model(
    model=dense_model,
    model_name="simple-autoencoder-outlier-trading",
    model_file_path="simple_autoencoder.pt",
    metadata={"hidden_layer_size": hidden_layer_size},
)
experiment_tracker.finish_run()

training_loss_figure = plot_loss_curve(
    training_loss_history,
    "Learning Curve - Autoencoder Reconstruction",
    loss_series_name="Train MSE (log-return windows)",
)
training_loss_figure.show()

## Erro de reconstrucao ao longo do tempo e limiar de corte

Com o modelo ja treinado e o run do wandb encerrado, o escore de
anomalia de cada janela e calculado sobre `full_window_dataset`, que
cobre cronologicamente toda a serie (treino seguido de validacao). O
grafico abaixo mostra o erro de reconstrucao calculado para cada
janela, na ordem cronologica da serie real de retornos. A linha
horizontal tracejada marca o `anomaly_threshold`: o valor de corte,
definido pelo percentil `(1 - CONTAMINATION)` dos escores, que separa
o que o modelo considera comportamento normal do que e classificado
como outlier. Pontos acima da linha sao os outliers efetivamente
detectados, destacados em laranja.

In [ ]:
full_window_loader = DataLoader(
    full_window_dataset,
    batch_size=len(full_window_dataset),
    shuffle=False,
)
full_windows_tensor = next(iter(full_window_loader)).to(device)

reconstruction_scores = compute_reconstruction_scores(
    dense_model, full_windows_tensor
)
anomaly_threshold = np.percentile(
    reconstruction_scores, 100 * (1 - CONTAMINATION)
)
is_window_outlier = reconstruction_scores > anomaly_threshold

print(f"Limiar de anomalia: {anomaly_threshold:.6f}")
print(f"Janelas classificadas como outlier: {is_window_outlier.sum()}")

In [ ]:
reconstruction_dates = dates.values[window_end_indices]

reconstruction_figure = plot_reconstruction_error_with_threshold(
    reconstruction_dates,
    reconstruction_scores,
    is_window_outlier,
    anomaly_threshold,
    "Erro de reconstrucao por janela e limiar de anomalia",
)
reconstruction_figure.show()

## Alinhamento dos rotulos com a serie original

Como cada janela termina em uma posicao diferente da serie original,
a celula abaixo cria um array booleano do tamanho da serie (`is_outlier`),
marcando como outlier cada ponto correspondente ao final de uma janela
classificada como anomala pelo autoencoder, e um array `reconstruction_score`
com o erro de reconstrucao alinhado a cada ponto (`NaN` fora das
posicoes de fim de janela).

In [ ]:
is_outlier = np.zeros(number_of_observations, dtype=bool)
is_outlier[window_end_indices[is_window_outlier]] = True

reconstruction_score = np.full(number_of_observations, np.nan)
reconstruction_score[window_end_indices] = reconstruction_scores

## Gerador de sinais de compra e venda

O `TradingSignalGenerator`, importado de `pgl_utils.deep_learning`, e
um componente baseado em regras que consome a saida do autoencoder
(quais pontos sao outliers) e a converte em sinais acionaveis de
compra e venda. A logica utilizada e a seguinte:

- Um outlier que ocorre junto de uma queda relevante de preco no curto
  prazo e interpretado como uma possivel sobrevenda, gerando um sinal
  de **compra**.
- Um outlier que ocorre junto de uma alta relevante de preco no curto
  prazo e interpretado como uma possivel sobrecompra, gerando um sinal
  de **venda**.
- Outliers cuja variacao de preco associada e pequena, dentro de uma
  zona neutra, nao geram sinal (`hold`).

Pontos que nao foram classificados como outliers pelo autoencoder
nunca geram sinal, ja que o gerador atua apenas sobre as anomalias
detectadas.

A celula abaixo instancia o gerador de sinais e gera os sinais de
compra e venda a partir dos outliers detectados pelo autoencoder
denso, sobre a serie de precos reais do ativo.

In [ ]:
signal_generator = TradingSignalGenerator(momentum_window=3, neutral_zone=0.01)
signal = signal_generator.generate_signals(price_values, is_outlier)

buy_indices = np.where(signal == "buy")[0]
sell_indices = np.where(signal == "sell")[0]

print(f"Sinais de compra gerados: {len(buy_indices)}")
print(f"Sinais de venda gerados: {len(sell_indices)}")

## Visualizacao dos outliers e das oportunidades de compra e venda

Os graficos a seguir apresentam, sobre a serie de precos reais do
ativo:

1. Os outliers detectados pelo autoencoder denso.
2. Os sinais de compra e venda gerados pelo `TradingSignalGenerator`
   a partir desses outliers.

Diferentemente do notebook com dados simulados, nao ha aqui um painel
de gabarito, pois os dados sao reais e nao existe um conjunto
conhecido de choques injetados. As contagens de outliers e de sinais
ja foram impressas apos o treinamento e nao sao reenviadas ao wandb,
pois o run foi encerrado logo apos a etapa de treinamento e validacao,
seguindo o mesmo padrao do `WandbExperimentTracker` usado em
`regression_log_return_training.ipynb`.

In [ ]:
outlier_indices = np.where(is_outlier)[0]

signals_figure = plot_outlier_detection_and_trading_signals(
    dates.values,
    price_values,
    outlier_indices,
    buy_indices,
    sell_indices,
    f"{TICKER} - deteccao de outliers (Autoencoder Simples)",
    "Oportunidades identificadas pelo TradingSignalGenerator",
)
signals_figure.update_layout(height=800)
signals_figure.show()

## Series sinteticas de log-retorno com GARCH(1,1)

Como complemento ao restante do notebook, esta secao usa um modelo
GARCH(1,1) para gerar series sinteticas de log-retorno a partir dos
mesmos dados reais do ativo ja coletados anteriormente (`price_tensor`
e `return_values`), sem a necessidade de baixar a serie novamente.

O objetivo e diferente da deteccao de outliers feita acima: em vez de
identificar anomalias na serie real, ajustamos um modelo GARCH(1,1)
com residuos t-Student sobre os log-retornos reais e simulamos
multiplas trajetorias futuras que preservam a mesma volatilidade
condicional e a mesma cauda pesada estimadas a partir dos dados reais.
Essas trajetorias sinteticas nunca ocorreram de fato, mas sao
estatisticamente compativeis com o processo estimado.

### Instalacao e importacao das bibliotecas do GARCH

A celula abaixo instala a biblioteca `arch`, usada para ajustar e
simular modelos GARCH, e importa os componentes necessarios para o
restante desta secao.

In [ ]:
!uv pip install arch -qqq

In [ ]:
from arch import arch_model
from arch.univariate import StudentsT

from pgl_utils.deep_learning import plot_real_and_synthetic_continuation

### Parametros da simulacao

As constantes abaixo controlam o horizonte de simulacao (em dias
uteis), o numero de trajetorias sinteticas geradas e quantas dessas
trajetorias sao efetivamente desenhadas nos graficos. A mesma
`RANDOM_SEED` definida no inicio do notebook e reutilizada para
garantir reprodutibilidade.

In [ ]:
SYNTHETIC_HORIZON_DAYS = 90
N_SYNTHETIC_PATHS = 200
N_PATHS_TO_PLOT = 30

### Ajuste do modelo GARCH(1,1)

A funcao abaixo ajusta um modelo GARCH(1,1) com residuos com
distribuicao t-Student sobre a serie de log-retornos, informada em
porcentagem (pratica usual ao utilizar a biblioteca `arch`). A semente
aleatoria e fixada diretamente na distribuicao dos residuos, pois e
esse objeto que controla a aleatoriedade utilizada posteriormente
durante a simulacao das trajetorias sinteticas.

In [ ]:
def fit_garch(
    log_returns_pct: np.ndarray,
    random_seed: int,
):
    """Fit a GARCH(1,1) model with Student's t errors.

    Args:
        log_returns_pct: One-dimensional array with the log-return
            series, expressed in percentage points.
        random_seed: Seed used to initialize the random number
            generator of the Student's t error distribution, ensuring
            that later simulations are reproducible.

    Returns:
        The fitted result object returned by the `arch` library, which
        exposes methods such as `.forecast()` and `.summary()`.
    """
    model_specification = arch_model(
        log_returns_pct,
        mean="Constant",
        vol="Garch",
        p=1,
        q=1,
        dist="t",
    )
    model_specification.distribution = StudentsT(seed=random_seed)
    fitted_result = model_specification.fit(disp="off")
    return fitted_result

In [ ]:
garch_result = fit_garch(log_return_tensor.numpy().astype(np.float64) * 100.0, RANDOM_SEED)

### Simulacao de trajetorias sinteticas de log-retorno

A partir do modelo GARCH ajustado, simulamos multiplas trajetorias
futuras de log-retorno. Cada trajetoria e gerada respeitando a mesma
estrutura de volatilidade condicional e a mesma distribuicao de cauda
pesada estimadas a partir dos dados reais.

In [ ]:
def simulate_synthetic_returns(
    garch_result,
    horizon_days: int,
    n_paths: int,
) -> np.ndarray:
    """Simulate synthetic log-return paths from a fitted GARCH model.

    Args:
        garch_result: Fitted GARCH result object, as returned by
            `fit_garch`.
        horizon_days: Number of future days to simulate for each path.
        n_paths: Number of independent synthetic paths to simulate.

    Returns:
        A two-dimensional array of shape (n_paths, horizon_days) with
        the simulated log-return paths, already converted back from
        percentage points to plain log-return units.
    """
    forecast_result = garch_result.forecast(
        horizon=horizon_days,
        method="simulation",
        simulations=n_paths,
        reindex=False,
    )
    synthetic_paths_pct = forecast_result.simulations.values[0]
    return synthetic_paths_pct / 100.0

In [ ]:
synthetic_returns = simulate_synthetic_returns(
    garch_result,
    SYNTHETIC_HORIZON_DAYS,
    N_SYNTHETIC_PATHS,
)

synthetic_dates = pd.bdate_range(
    start=dates[-1] + pd.Timedelta(days=1),
    periods=SYNTHETIC_HORIZON_DAYS,
)

synthetic_returns.shape

### Reconstrucao do preco sintetico

Cada trajetoria sintetica de log-retorno pode ser transformada de
volta em uma trajetoria de preco, encadeando os retornos a partir do
ultimo preco real conhecido: `P_sintetico[t+1] = P[t] * exp(r[t+1])`.

In [ ]:
def reconstruct_price_paths(
    last_known_price: float,
    synthetic_returns: np.ndarray,
) -> np.ndarray:
    """Reconstruct synthetic price paths from simulated log-returns.

    Args:
        last_known_price: Last observed real price, used as the
            starting point for every reconstructed path.
        synthetic_returns: Two-dimensional array of shape
            (n_paths, horizon_days) with the simulated log-return
            paths.

    Returns:
        A two-dimensional array with the same shape as
        `synthetic_returns`, containing the reconstructed synthetic
        price paths.
    """
    cumulative_log_return = np.cumsum(synthetic_returns, axis=1)
    return last_known_price * np.exp(cumulative_log_return)

In [ ]:
last_known_price = price_values[-1]
synthetic_prices = reconstruct_price_paths(last_known_price, synthetic_returns)

mean_synthetic_return_path = synthetic_returns.mean(axis=0)
mean_synthetic_price_path = synthetic_prices.mean(axis=0)

### Visualizacao: serie real seguida da continuacao sintetica

`plot_real_and_synthetic_continuation` (importada de
`pgl_utils.deep_learning`) retorna tres graficos:

1. Um grafico combinado, em dois paineis (log-retorno e preco), com a
   serie real seguida de todas as trajetorias sinteticas simuladas
   sobrepostas e sua media.
2. Um grid com uma trajetoria sintetica de log-retorno por painel,
   cada uma precedida pelos ultimos `tail_real_days_for_grid` dias da
   serie real (por padrao, os ultimos 30 dias, para simplificar o
   grafico).
3. O mesmo grid, para os precos sinteticos reconstruidos.

In [ ]:
synthetic_figure, returns_grid_figure, prices_grid_figure = (
    plot_real_and_synthetic_continuation(
        dates.values,
        return_values,
        dates.values,
        price_values,
        synthetic_dates,
        synthetic_returns,
        synthetic_prices,
        mean_synthetic_return_path,
        mean_synthetic_price_path,
        TICKER,
        n_paths_to_plot=N_PATHS_TO_PLOT,
    )
)
# synthetic_figure.show()
returns_grid_figure.show()
prices_grid_figure.show()

## Avaliacao de pontos de compra e venda nas series sinteticas (GARCH)

Nesta secao, reaproveitamos o autoencoder `dense_model`, ja treinado
sobre os log-retornos reais, e o `signal_generator`
(`TradingSignalGenerator`) definido anteriormente, para avaliar os
pontos de compra e venda diretamente sobre as `N_SYNTHETIC_PATHS`
trajetorias sinteticas geradas pelo GARCH(1,1). Nenhum modelo e
retreinado: o mesmo autoencoder e o mesmo limiar de anomalia
(`anomaly_threshold`), aprendidos a partir dos dados reais, sao
aplicados as series sinteticas.

### Normalizacao das series sinteticas

Os log-retornos sinteticos sao normalizados com a mesma media e desvio
padrao do conjunto de treino real (`return_train_mean` e
`return_train_std`), evitando qualquer vazamento de informacao e
garantindo que o autoencoder receba entradas na mesma escala usada
durante o treinamento.

In [ ]:
synthetic_returns_tensor = torch.from_numpy(synthetic_returns.astype(np.float32))
normalized_synthetic_returns_tensor, _, _ = normalize_series(
    synthetic_returns_tensor,
    mean=return_train_mean,
    std=return_train_std,
)

### Construcao das janelas deslizantes por trajetoria sintetica

Cada uma das `N_SYNTHETIC_PATHS` trajetorias sinteticas e tratada de
forma independente: a funcao abaixo constroi, para cada trajetoria,
todas as janelas deslizantes de tamanho `WINDOW_SIZE`, da mesma forma
que foi feito para a serie real, mas de uma vez para o lote inteiro de
trajetorias.

In [ ]:
def build_sliding_windows(
    series_values: np.ndarray,
    window_size: int,
) -> np.ndarray:
    """Build sliding windows along the last axis of a batch of series.

    Args:
        series_values: Array of shape (n_series, horizon_days) with
            one or more time series.
        window_size: Number of consecutive time steps contained in
            each window.

    Returns:
        An array of shape (n_series, horizon_days - window_size + 1,
        window_size) with every sliding window of each series.
    """
    return np.lib.stride_tricks.sliding_window_view(
        series_values, window_shape=window_size, axis=1,
    )

In [ ]:
synthetic_windows = build_sliding_windows(
    normalized_synthetic_returns_tensor.numpy(), WINDOW_SIZE,
)
n_synthetic_paths, n_windows_per_path, _ = synthetic_windows.shape

synthetic_windows_tensor = torch.from_numpy(
    synthetic_windows.reshape(-1, WINDOW_SIZE)
).to(device)

synthetic_windows_tensor.shape

### Erro de reconstrucao e deteccao de outliers nas series sinteticas

O erro de reconstrucao de cada janela sintetica e calculado com a
mesma funcao `compute_reconstruction_scores` usada para a serie real.
Para manter a mesma nocao de anomalia aprendida pelo autoencoder,
reaproveitamos o `anomaly_threshold` ja calculado a partir dos dados
reais, em vez de recalcular um novo limiar especificamente para as
series sinteticas.

In [ ]:
synthetic_reconstruction_scores = compute_reconstruction_scores(
    dense_model, synthetic_windows_tensor
).reshape(n_synthetic_paths, n_windows_per_path)

is_synthetic_window_outlier = synthetic_reconstruction_scores > anomaly_threshold

print(
    "Media de janelas-outlier por trajetoria sintetica: "
    f"{is_synthetic_window_outlier.sum(axis=1).mean():.2f} de "
    f"{n_windows_per_path}"
)

### Sinais de compra e venda nas series sinteticas

Os outliers de cada trajetoria sao alinhados a serie sintetica de
origem (mesmo procedimento aplicado a serie real) e convertidos em
sinais de compra e venda pelo mesmo `signal_generator`
(`TradingSignalGenerator`), usando o preco sintetico reconstruido de
cada trajetoria.

In [ ]:
synthetic_window_end_indices = np.arange(WINDOW_SIZE - 1, SYNTHETIC_HORIZON_DAYS)

synthetic_point_outlier = np.zeros(
    (n_synthetic_paths, SYNTHETIC_HORIZON_DAYS), dtype=bool
)
synthetic_point_outlier[:, synthetic_window_end_indices] = is_synthetic_window_outlier

synthetic_signals = np.full(
    (n_synthetic_paths, SYNTHETIC_HORIZON_DAYS), "hold", dtype=object
)
for path_index in range(n_synthetic_paths):
    synthetic_signals[path_index] = signal_generator.generate_signals(
        synthetic_prices[path_index], synthetic_point_outlier[path_index],
    )

synthetic_buy_counts = (synthetic_signals == "buy").sum(axis=1)
synthetic_sell_counts = (synthetic_signals == "sell").sum(axis=1)

print(f"Sinais de compra por trajetoria (media): {synthetic_buy_counts.mean():.2f}")
print(f"Sinais de venda por trajetoria (media): {synthetic_sell_counts.mean():.2f}")
print(
    "Trajetorias sinteticas sem nenhum sinal: "
    f"{((synthetic_buy_counts + synthetic_sell_counts) == 0).sum()} de "
    f"{n_synthetic_paths}"
)

### Exemplo: outliers e sinais em uma trajetoria sintetica

Para ilustrar o resultado, escolhemos a trajetoria sintetica com o
maior numero total de sinais (compra + venda) e reaproveitamos
`plot_outlier_detection_and_trading_signals` para visualiza-la, da
mesma forma que foi feito para a serie real.

In [ ]:
example_path_index = int(np.argmax(synthetic_buy_counts + synthetic_sell_counts))

example_outlier_indices = np.where(synthetic_point_outlier[example_path_index])[0]
example_buy_indices = np.where(synthetic_signals[example_path_index] == "buy")[0]
example_sell_indices = np.where(synthetic_signals[example_path_index] == "sell")[0]

synthetic_signals_figure = plot_outlier_detection_and_trading_signals(
    synthetic_dates,
    synthetic_prices[example_path_index],
    example_outlier_indices,
    example_buy_indices,
    example_sell_indices,
    f"{TICKER} - outliers na trajetoria sintetica {example_path_index} (GARCH)",
    f"Sinais de compra/venda na trajetoria sintetica {example_path_index}",
)
synthetic_signals_figure.update_layout(height=800)
synthetic_signals_figure.show()